# Lab | Data Structuring and Combining Data - Solution Notebook

---
# Challenge 1: Combining & Cleaning Data

**Goal**: Load 3 CSV files, combine them into one DataFrame, and clean the combined dataset.

**Files**:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Step 1: Load all three files separately
df1 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
df2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv")
df3 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv")

print("file1 shape:", df1.shape)
print("file2 shape:", df2.shape)
print("file3 shape:", df3.shape)

In [ ]:
# Step 2: Check if all files have the same columns
print("file1 columns:", df1.columns.tolist())
print()
print("file2 columns:", df2.columns.tolist())
print()
print("file3 columns:", df3.columns.tolist())

In [ ]:
# Step 3: Combine all three DataFrames using pd.concat()
# pd.concat() stacks DataFrames on top of each other (axis=0 means vertically)
# ignore_index=True resets the index so it runs from 0 to the end
combined_df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

print("Combined shape:", combined_df.shape)
combined_df.head()

In [ ]:
# Step 4: Clean the combined dataset
# Apply all the same cleaning steps as the previous lab

# 4a: Clean column names
combined_df.columns = [col.lower().replace(' ', '_') for col in combined_df.columns]
combined_df = combined_df.rename(columns={'st': 'state'})

print("Columns after cleaning:", combined_df.columns.tolist())

In [ ]:
# 4b: Clean invalid values

# Fix gender
combined_df['gender'] = combined_df['gender'].replace({
    'Femal': 'F', 'female': 'F', 'Male': 'M'
})

# Fix state
combined_df['state'] = combined_df['state'].replace({
    'AZ': 'Arizona', 'Cali': 'California', 'WA': 'Washington'
})

# Fix education
combined_df['education'] = combined_df['education'].replace({'Bachelors': 'Bachelor'})

# Remove % from customer_lifetime_value
combined_df['customer_lifetime_value'] = combined_df['customer_lifetime_value'].str.replace('%', '')

# Fix vehicle_class
combined_df['vehicle_class'] = combined_df['vehicle_class'].replace({
    'Sports Car': 'Luxury', 'Luxury SUV': 'Luxury', 'Luxury Car': 'Luxury'
})

print("Invalid values cleaned.")

In [ ]:
# 4c: Fix data types

# Convert customer_lifetime_value to float
combined_df['customer_lifetime_value'] = combined_df['customer_lifetime_value'].astype(float)

# Fix number_of_open_complaints — extract middle number from '1/0/00' format
combined_df['number_of_open_complaints'] = (
    combined_df['number_of_open_complaints'].str.split('/').str[1].astype(float)
)

print("Data types fixed.")
print(combined_df.dtypes)

In [ ]:
# 4d: Handle null values

# Drop rows where ALL values are null
combined_df = combined_df.dropna(how='all')

# Fill numerical nulls with mean
numerical_cols = ['customer_lifetime_value', 'income', 'monthly_premium_auto',
                  'number_of_open_complaints', 'total_claim_amount']
for col in numerical_cols:
    combined_df[col] = combined_df[col].fillna(combined_df[col].mean())

# Fill categorical nulls with mode
categorical_cols = ['gender', 'state', 'education', 'policy_type', 'vehicle_class']
for col in categorical_cols:
    combined_df[col] = combined_df[col].fillna(combined_df[col].mode()[0])

print("Nulls remaining:", combined_df.isnull().sum().sum())

In [ ]:
# 4e: Drop duplicates and reset index
combined_df = combined_df.drop_duplicates(keep='first').reset_index(drop=True)

print("Final combined dataset shape:", combined_df.shape)
combined_df.head()

---
# Challenge 2: Structuring Data

**Dataset**: marketing_customer_analysis_clean.csv (already clean)

**Source**: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

In [ ]:
# Load the clean marketing dataset
marketing_df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

print("Shape:", marketing_df.shape)
marketing_df.head()

### Exercise 1: Total Revenue by Sales Channel (Pivot Table)

**Goal**: Find which sales channel brought the most total revenue using a pivot table.

In [ ]:
# Create a pivot table showing total revenue per sales channel
# values = the column we want to aggregate (total_claim_amount = revenue)
# index = what to group by (sales_channel)
# aggfunc = how to aggregate (sum = add all values together)
revenue_pivot = marketing_df.pivot_table(
    values='total_claim_amount',
    index='sales_channel',
    aggfunc='sum'
).round(2)

# Sort from highest to lowest revenue
revenue_pivot = revenue_pivot.sort_values('total_claim_amount', ascending=False)

print("Total Revenue by Sales Channel:")
print(revenue_pivot)

**Insights**:
- The sales channel at the top of the table brought the most total revenue
- This channel should be prioritized for investment and scaling
- Lower-performing channels may need strategic review or more support

### Exercise 2: Average CLV by Gender and Education Level (Pivot Table)

**Goal**: See how customer lifetime value varies across gender and education combinations.

In [ ]:
# Create a pivot table with gender as rows and education as columns
# values = customer_lifetime_value
# index = gender (rows)
# columns = education (columns)
# aggfunc = mean (average CLV for each combination)
clv_pivot = marketing_df.pivot_table(
    values='customer_lifetime_value',
    index='gender',
    columns='education',
    aggfunc='mean'
).round(2)

print("Average Customer Lifetime Value by Gender and Education:")
print(clv_pivot)

**Insights**:
- Each cell shows the average CLV for that gender + education combination
- Higher CLV segments are more valuable long-term customers
- Marketing campaigns can be targeted at high-CLV segments for maximum return
- Compare across rows (gender) and columns (education) to spot patterns

---
## Bonus: Complaints by Policy Type and Month (Long Format)

**Goal**: Show number of complaints by policy type and month in long format.

In [ ]:
# Step 1: Extract month from date column
marketing_df['month'] = pd.to_datetime(marketing_df['effective_to_date']).dt.month_name()

# Step 2: Create pivot table - complaints by policy type and month
# number_of_open_complaints is our measure
complaints_pivot = marketing_df.pivot_table(
    values='number_of_open_complaints',
    index='policy_type',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print("Complaints by Policy Type and Month (Wide Format):")
print(complaints_pivot)

In [ ]:
# Step 3: Convert to long format using melt()
# Wide format: each month is a column
# Long format: each row is one policy_type + month combination

# melt() unpivots the DataFrame
# id_vars = columns to keep as identifiers
# var_name = name for the new column that holds the old column names
# value_name = name for the new column that holds the values
complaints_long = complaints_pivot.reset_index().melt(
    id_vars='policy_type',
    var_name='month',
    value_name='number_of_complaints'
)

# Sort by policy type and complaints descending
complaints_long = complaints_long.sort_values(
    ['policy_type', 'number_of_complaints'],
    ascending=[True, False]
).reset_index(drop=True)

print("Complaints by Policy Type and Month (Long Format):")
print(complaints_long)

**Why Long Format?**

Long format is preferred for:
- Easier filtering and grouping
- Better compatibility with visualization libraries
- Each row represents one observation (policy type + month + count)

**Insight**: The month with the highest complaints for each policy type can be identified from this table, helping the customer service team plan resources.